# TSTR vs TRTR Evaluation Pipeline

**Strategy:** This notebook compares a model trained on Real-only data (TRTR) versus Real + Synthetic data (TSTR).

**Pipeline:**
1. Hyperparameter tuning
2. Cross-validation comparisons
3. Outlier fold investigation
4. Final test set evaluation
5. Feature importance analysis
6. Deployment CSV export



## Imports & Configuration



In [1]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))

from itertools import product
from typing import Dict, List, Optional
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score, f1_score, recall_score, precision_score, accuracy_score
from src.C45DecisionTree import C45DecisionTree
from pathlib import Path



## Feature + Domain Definitions



In [2]:
FUNA_DB_DOMAIN_MAPPING: Dict[str, str] = {
    "NC":  "Number Comparison",
    "DM":  "Digit-Dot Matching",
    "NP":  "Overall Processing Efficiency",
    "SN":  "Symbolic vs. Non-Symbolic Processing Difference",
    "NS":  "Number Series",
    "ADD": "Single-Digit Addition",
    "SUB": "Single-Digit Subtraction",
    "CA":  "Multi-Digit Addition and Subtraction",
    "AF":  "Overall Arithmetic Fluency",
    "BC":  "Basic vs. Complex Arithmetic Contrast",
    "AS":  "Addition vs. Subtraction Asymmetry",
    "PF":  "Processing-Fluency Integration",
}
FUNA_DB_RAW_FEATURES: List[str] = [
    "NC", "DM", "NS", "ADD", "SUB", "CA",
    "NC_incomplete", "DM_incomplete", "NS_incomplete",
    "ADD_incomplete", "SUB_incomplete", "CA_incomplete",
]
FUNA_DB_DIAGNOSTIC_FEATURES: List[str] = [
    "NC", "DM", "NS", "ADD", "SUB", "CA",
    "NP", "SN", "AF", "BC", "AS", "PF",
    "NC_incomplete", "DM_incomplete", "NS_incomplete",
    "ADD_incomplete", "SUB_incomplete", "CA_incomplete",
]

# Features that signal missing/incomplete data.
# The tree may still split on these (they carry real signal),
# but we exclude them from diagnostic z-score severity so that
# "timed out" is never conflated with "performed poorly".
INCOMPLETE_FLAGS = {
    "NC_incomplete", "DM_incomplete", "NS_incomplete",
    "ADD_incomplete", "SUB_incomplete", "CA_incomplete"
}




## Helpers



In [3]:

def split_xy(df: pd.DataFrame, label_col: str = 'Label'):
    X = df[FUNA_DB_DIAGNOSTIC_FEATURES]
    y = df[label_col]
    return X, y

def get_preds(tree, X: pd.DataFrame) -> List[int]:
    return [int(pred) for pred in tree.predict(X)]

def compute_metrics(y_true, preds) -> dict:
    return {
        'f1':        f1_score(y_true, preds, zero_division=0),
        'fbeta':     fbeta_score(y_true, preds, beta=2, zero_division=0),
        'recall':    recall_score(y_true, preds, zero_division=0),
        'precision': precision_score(y_true, preds, zero_division=0),
        'accuracy':  accuracy_score(y_true, preds),
    }

def run_cv(X_train: pd.DataFrame, y_train: pd.Series, label: str,
           best_params: dict,
           n_real: Optional[int] = None) -> tuple[dict, list[dict]]:

    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_f1, cv_fbeta, cv_recall, cv_precision, cv_accuracy = [], [], [], [], []
    fold_records = []

    if n_real is not None:
        X_real  = X_train.iloc[:n_real].reset_index(drop=True)
        y_real  = y_train.iloc[:n_real].reset_index(drop=True)
        X_synth = X_train.iloc[n_real:].reset_index(drop=True)
        y_synth = y_train.iloc[n_real:].reset_index(drop=True)
    else:
        X_real, y_real = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
        X_synth, y_synth = X_train.iloc[:0], y_train.iloc[:0]

    print(f'\nStarting 5-Fold Cross-Validation [{label}]...')
    if n_real is not None:
        print(f'  (folding over {len(X_real)} real samples; '
              f'{len(X_synth)} synthetic rows pinned to every train fold)')

    for fold, (train_index, val_index) in enumerate(skf.split(X_real, y_real), 1):

        X_val_fold = X_real.iloc[val_index]
        y_val_fold = y_real.iloc[val_index]

        X_trn_fold = pd.concat([X_real.iloc[train_index], X_synth], ignore_index=True)
        y_trn_fold = pd.concat([y_real.iloc[train_index], y_synth], ignore_index=True)

        tree = C45DecisionTree(**best_params, feature_domain_mapping=FUNA_DB_DOMAIN_MAPPING)
        tree.fit(X_trn_fold, y_trn_fold,
                 raw_features=FUNA_DB_RAW_FEATURES)

        preds = get_preds(tree, X_val_fold)
        m     = compute_metrics(y_val_fold, preds)

        cv_f1.append(m['f1'])
        cv_fbeta.append(m['fbeta'])
        cv_recall.append(m['recall'])
        cv_precision.append(m['precision'])
        cv_accuracy.append(m['accuracy'])

        fold_records.append({
            'fold': fold,
            'recall': m['recall'],
            'fbeta': m['fbeta'],
            'tree_depth': tree.get_depth(),
            'tree_leaves': tree.get_leaves_num(),
            'n_train':       len(X_trn_fold),
            'class_0_train': int((y_trn_fold == 0).sum()),
            'class_1_train': int((y_trn_fold == 1).sum()),
            'n_val':         len(X_val_fold),
            'class_0_val':   int((y_val_fold == 0).sum()),
            'class_1_val':   int((y_val_fold == 1).sum()),
        })

        print(f'  Fold {fold}: F1={m["f1"]:.4f}, F2={m["fbeta"]:.4f}, Recall={m["recall"]:.4f}')

    aggregated = {
        'mean_f1': np.mean(cv_f1), 'std_f1': np.std(cv_f1),
        'mean_fbeta': np.mean(cv_fbeta), 'std_fbeta': np.std(cv_fbeta),
        'mean_recall': np.mean(cv_recall), 'std_recall': np.std(cv_recall),
        'mean_precision': np.mean(cv_precision), 'std_precision': np.std(cv_precision),
        'mean_accuracy': np.mean(cv_accuracy), 'std_accuracy': np.std(cv_accuracy),
    }

    return aggregated, fold_records

def print_cv_results(label: str, metrics: dict):
    print(f'\n=== Cross-Validation Performance [{label}] (5-Fold) ===')
    print(f'  Recall:    {metrics["mean_recall"]:.4f} +/- {metrics["std_recall"]:.4f}')
    print(f'  Precision: {metrics["mean_precision"]:.4f} +/- {metrics["std_precision"]:.4f}')
    print(f'  F1-Score:  {metrics["mean_f1"]:.4f} +/- {metrics["std_f1"]:.4f}')
    print(f'  F2-Score:  {metrics["mean_fbeta"]:.4f} +/- {metrics["std_fbeta"]:.4f}')
    print(f'  Accuracy:  {metrics["mean_accuracy"]:.4f} +/- {metrics["std_accuracy"]:.4f}')

def print_variance_inspection(label: str, fold_records: list[dict]):
    df = pd.DataFrame(fold_records)
    mean_recall = df['recall'].mean()
    std_recall  = df['recall'].std()

    print(f'\n--- Fold Variance Inspection [{label}] ---')
    print(f'{"Fold":>5} {"N_train":>8} {"C0_tr":>6} {"C1_tr":>6} '
          f'{"N_val":>6} {"C0_val":>7} {"C1_val":>7} '
          f'{"Depth":>6} {"Leaves":>7} {"Recall":>8} {"F2":>8} {"Flag"}')
    print('-' * 95)

    for _, row in df.iterrows():
        flag = (
            ' HIGH' if row['recall'] > mean_recall + std_recall else
            ' LOW' if row['recall'] < mean_recall - std_recall else
            ''
        )
        print(f'{int(row["fold"]):>5} {int(row["n_train"]):>8} {int(row["class_0_train"]):>6} '
              f'{int(row["class_1_train"]):>6} {int(row["n_val"]):>6} '
              f'{int(row["class_0_val"]):>7} {int(row["class_1_val"]):>7} '
              f'{int(row["tree_depth"]):>6} {int(row["tree_leaves"]):>7} '
              f'{row["recall"]:>8.4f} {row["fbeta"]:>8.4f}{flag}')

    print(f'\n  Recall range : {df["recall"].min():.4f} - {df["recall"].max():.4f}')
    print(f'  Recall mean  : {mean_recall:.4f} +/- {std_recall:.4f}')

    corr_c1    = df['class_1_train'].corr(df['recall'])
    corr_depth = df['tree_depth'].corr(df['recall'])
    print(f'\n  Correlation: class_1_train vs recall = {corr_c1:+.3f}')
    print(f'  Correlation: tree_depth     vs recall = {corr_depth:+.3f}')
    if abs(corr_c1) > 0.6:
        print('  Strong correlation with positive-class count in training fold.')
        print('    Synthetic data may not be uniformly covering all subgroups.')




## Load Data



In [4]:
print('Loading 70/15/15 Datasets...')
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR.parent / "datasets" / "processed"

r_train = pd.read_csv(DATASET_DIR / "train.csv")
s_train = pd.read_csv(DATASET_DIR / "s_train.csv")
val_df  = pd.read_csv(DATASET_DIR / "val.csv")
test_df = pd.read_csv(DATASET_DIR / "test.csv")




Loading 70/15/15 Datasets...


## Prepare Train Sets



In [5]:
train_trtr = r_train.copy()                                     # TRTR: real only
train_tstr = pd.concat([r_train, s_train], ignore_index=True)  # TSTR: real + synthetic

print(f'TRTR Train shape (Real 70%):         {train_trtr.shape}')
print(f'TSTR Train shape (Real + Synth):     {train_tstr.shape}')
print(f'Validation shape (Real 15%):         {val_df.shape}')
print(f'Test shape (Real 15%):               {test_df.shape}')

print(f'\nSynthetic data class distribution:')
print(s_train['Label'].value_counts().rename({0: 'Typical (0)', 1: 'At-Risk (1)'}).to_string())




TRTR Train shape (Real 70%):         (250, 19)
TSTR Train shape (Real + Synth):     (308, 19)
Validation shape (Real 15%):         (54, 19)
Test shape (Real 15%):               (54, 19)

Synthetic data class distribution:
Label
At-Risk (1)    58


## Split X / y



In [6]:
X_train_trtr, y_train_trtr = split_xy(train_trtr)
X_train_tstr, y_train_tstr = split_xy(train_tstr)
X_val,  y_val  = split_xy(val_df)
X_test, y_test = split_xy(test_df)




## Hyperparameter Search Space



In [7]:
PARAM_GRID = {
    # CF in [0.10, 0.50]; sampled because confidence factor is continuous.
    'conf_fact':        np.round(np.arange(0.10, 0.51, 0.05), 2).tolist(),
    # n_min in {x | x in N, 10 <= x <= 50}
    'min_samples_leaf': list(range(10, 51)),
    # d_max in {x | x in N, 5 <= x <= 15}
    'max_depth':        list(range(5, 16)),
}

def iter_param_grid(param_grid: dict):
    keys = list(param_grid.keys())
    for values in product(*(param_grid[k] for k in keys)):
        yield dict(zip(keys, values))




## PHASE 1 — Hyperparameter Search

Train one tree per hyperparameter candidate and evaluate the tree's native class predictions. The selected setting is the best hyperparameter set by validation F2.



In [ ]:
from joblib import Parallel, delayed

# Use process-based parallelism for the custom Python tree code.
N_JOBS = min(os.cpu_count() or 1, 12)
JOBLIB_BACKEND = 'loky'
GRID_SEARCH_OUTPUT_DIR = BASE_DIR.parent / 'outputs' / 'grid_search'
GRID_SEARCH_EXPORT_COLS = [
    'fbeta', 'recall', 'precision', 'f1', 'accuracy',
    *PARAM_GRID.keys(), 'tree_depth', 'tree_leaves'
]
VALIDATION_SELECTION_METRIC_COLS = ['fbeta', 'recall', 'precision', 'f1', 'accuracy']
BEST_SELECTION_SORT_COLS = [
    *VALIDATION_SELECTION_METRIC_COLS,
]
TIE_RESOLUTION_SORT_COLS = [
    'cv_mean_fbeta', 'cv_mean_recall', 'cv_mean_precision', 'cv_mean_f1', 'cv_mean_accuracy',
    *PARAM_GRID.keys(), 'tree_depth', 'tree_leaves'
]

print('\n' + '=' * 60)
print('PHASE 1: Hyperparameter Search')
print('=' * 60)

print('\nSearch grid:')
for param_name, values in PARAM_GRID.items():
    print(f'  {param_name}: {values}')
print(f'  parallel jobs: {N_JOBS} ({JOBLIB_BACKEND})')

def _evaluate_candidate(candidate_no, params, X_train, y_train, X_val, y_val):
    tree = C45DecisionTree(**params, feature_domain_mapping=FUNA_DB_DOMAIN_MAPPING)
    tree.fit(X_train, y_train, raw_features=FUNA_DB_RAW_FEATURES)

    preds = get_preds(tree, X_val)
    metrics = compute_metrics(y_val, preds)

    return {
        **params,
        'candidate_no': candidate_no,
        'f1': metrics['f1'],
        'fbeta': metrics['fbeta'],
        'recall': metrics['recall'],
        'precision': metrics['precision'],
        'accuracy': metrics['accuracy'],
        'tree_depth': tree.get_depth(),
        'tree_leaves': tree.get_leaves_num(),
    }

def _as_python_scalar(value):
    return value.item() if hasattr(value, 'item') else value

def save_grid_search_results(search_df, label):
    GRID_SEARCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = GRID_SEARCH_OUTPUT_DIR / f'{label.lower()}_no_threshold_grid_search_results.csv'
    search_df[GRID_SEARCH_EXPORT_COLS].to_csv(output_path, index=False)
    print(f'[{label}] Saved full grid search results ({len(search_df)} rows): {output_path}')
    return output_path

def _params_from_row(row):
    params = {k: _as_python_scalar(row[k]) for k in PARAM_GRID.keys()}
    params['min_samples_leaf'] = int(params['min_samples_leaf'])
    params['max_depth'] = int(params['max_depth'])
    return params

def _validation_best_ties(search_df):
    top_row = search_df.iloc[0]
    tied_mask = np.logical_and.reduce([
        np.isclose(search_df[col], top_row[col])
        for col in VALIDATION_SELECTION_METRIC_COLS
    ])
    tied_cols = [*VALIDATION_SELECTION_METRIC_COLS, *PARAM_GRID.keys(), 'tree_depth', 'tree_leaves']
    return search_df.loc[tied_mask, tied_cols].drop_duplicates(
        subset=[*PARAM_GRID.keys(), 'tree_depth', 'tree_leaves']
    ).reset_index(drop=True)

def _evaluate_tied_candidate(row, X_train, y_train, label, n_real=None):
    params = _params_from_row(row)

    cv_metrics, _ = run_cv(X_train, y_train, label, params, n_real=n_real)

    return {
        **{f'val_{col}': float(row[col]) for col in VALIDATION_SELECTION_METRIC_COLS},
        **{f'cv_{key}': float(value) for key, value in cv_metrics.items()},
        **params,
        'tree_depth': int(row['tree_depth']),
        'tree_leaves': int(row['tree_leaves']),
    }

def resolve_validation_ties_with_cv(search_df, X_train, y_train, label, n_real=None):
    tied_df = _validation_best_ties(search_df)
    print(f'\n[{label}] Validation-best tied candidates: {len(tied_df)}')

    tie_records = []
    for candidate_no, (_, row) in enumerate(tied_df.iterrows(), 1):
        print(f'\n[{label}] Tie candidate {candidate_no}/{len(tied_df)}: '
              f'params={_params_from_row(row)}')
        tie_records.append(_evaluate_tied_candidate(
            row, X_train, y_train, label, n_real=n_real
        ))

    tie_results_df = pd.DataFrame(tie_records).sort_values(
        by=TIE_RESOLUTION_SORT_COLS,
        ascending=False,
    )

    GRID_SEARCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = GRID_SEARCH_OUTPUT_DIR / f'{label.lower()}_validation_tie_no_threshold_cv_results.csv'
    tie_results_df.to_csv(output_path, index=False)
    print(f'[{label}] Saved validation-tie CV results ({len(tie_results_df)} rows): {output_path}')

    best_row = tie_results_df.iloc[0]
    best_params = _params_from_row(best_row)

    print(f'\n[{label}] Selected after validation-tie CV resolution:')
    print(f'  Params: {best_params}')
    print(f'  CV F2={best_row["cv_mean_fbeta"]:.4f}, '
          f'CV Recall={best_row["cv_mean_recall"]:.4f}, '
          f'CV Precision={best_row["cv_mean_precision"]:.4f}')

    return best_params, tie_results_df

def tune_params(X_train, y_train, X_val, y_val, label):
    candidates = list(enumerate(iter_param_grid(PARAM_GRID), 1))
    n_candidates = len(candidates)

    print(f'\n[{label}] Parallel search: {n_candidates} param combos')

    search_records = Parallel(
        n_jobs=N_JOBS,
        backend=JOBLIB_BACKEND,
        prefer='processes',
        verbose=10,
        batch_size=1,
        pre_dispatch='2*n_jobs',
    )(
        delayed(_evaluate_candidate)(candidate_no, params, X_train, y_train, X_val, y_val)
        for candidate_no, params in candidates
    )

    if not search_records:
        raise RuntimeError(f'[{label}] No search records were generated.')

    search_df = pd.DataFrame(search_records).sort_values(
        by=BEST_SELECTION_SORT_COLS,
        ascending=False,
    )

    save_grid_search_results(search_df, label)

    best_row = search_df.iloc[0]
    best_params = {k: _as_python_scalar(best_row[k]) for k in PARAM_GRID.keys()}
    best_params['min_samples_leaf'] = int(best_params['min_samples_leaf'])
    best_params['max_depth'] = int(best_params['max_depth'])

    best_metrics = {
        'f1': float(best_row['f1']),
        'fbeta': float(best_row['fbeta']),
        'recall': float(best_row['recall']),
        'precision': float(best_row['precision']),
        'accuracy': float(best_row['accuracy']),
    }

    print(f'\n[{label}] Top validation candidates:')
    print(search_df[GRID_SEARCH_EXPORT_COLS].head(10).to_string(index=False))

    print(f'\n[{label}] Selected validation setting:')
    print(f'  Params: {best_params}')
    print(f'  F2={best_metrics["fbeta"]:.4f}, '
          f'Recall={best_metrics["recall"]:.4f}, '
          f'Precision={best_metrics["precision"]:.4f}')

    return best_params, search_df

BEST_PARAMS_TRTR, SEARCH_RESULTS_TRTR = tune_params(
    X_train_trtr, y_train_trtr, X_val, y_val, 'TRTR'
)

BEST_PARAMS_TSTR, SEARCH_RESULTS_TSTR = tune_params(
    X_train_tstr, y_train_tstr, X_val, y_val, 'TSTR'
)

BEST_PARAMS_TRTR, TIE_RESULTS_TRTR = resolve_validation_ties_with_cv(
    SEARCH_RESULTS_TRTR, X_train_trtr, y_train_trtr, 'TRTR'
)

BEST_PARAMS_TSTR, TIE_RESULTS_TSTR = resolve_validation_ties_with_cv(
    SEARCH_RESULTS_TSTR, X_train_tstr, y_train_tstr, 'TSTR',
    n_real=len(r_train)
)





PHASE 1: Hyperparameter Search

Search grid:
  conf_fact: [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
  min_samples_leaf: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
  max_depth: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  parallel jobs: 12 (loky)

[TRTR] Parallel search: 4059 param combos


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   1 tasks      | elapsed:    4.2s
[Parallel(n_jobs=12)]: Done   8 tasks      | elapsed:    6.0s
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    9.3s
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:   11.8s
[Parallel(n_jobs=12)]: Done  37 tasks      | elapsed:   14.5s
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:   17.9s
[Parallel(n_jobs=12)]: Done  61 tasks      | elapsed:   21.8s
[Parallel(n_jobs=12)]: Done  74 tasks      | elapsed:   25.8s
[Parallel(n_jobs=12)]: Done  89 tasks      | elapsed:   29.0s
[Parallel(n_jobs=12)]: Done 104 tasks      | elapsed:   33.9s
[Parallel(n_jobs=12)]: Done 121 tasks      | elapsed:   37.9s
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:   41.8s
[Parallel(n_jobs=12)]: Done 157 tasks      | elapsed:   47.2s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:   51.8s
[Parallel(n_jobs=12)]: Done 197 tasks      | elapsed:  

[TRTR] Saved full grid search results (4059 rows): /home/caineirb/Documents/DysCalc/DysCalc-ML-Development/outputs/grid_search/trtr_no_threshold_grid_search_results.csv

[TRTR] Top validation candidates:
   fbeta   recall  precision      f1  accuracy  conf_fact  min_samples_leaf  max_depth  tree_depth  tree_leaves
0.779817 0.809524       0.68 0.73913  0.777778       0.25                10          5           5            8
0.779817 0.809524       0.68 0.73913  0.777778       0.25                10          6           5            8
0.779817 0.809524       0.68 0.73913  0.777778       0.25                10          7           5            8
0.779817 0.809524       0.68 0.73913  0.777778       0.25                10          8           5            8
0.779817 0.809524       0.68 0.73913  0.777778       0.25                10          9           5            8
0.779817 0.809524       0.68 0.73913  0.777778       0.25                10         10           5            8
0.779817 0.8

[Parallel(n_jobs=12)]: Done   1 tasks      | elapsed:    4.7s
[Parallel(n_jobs=12)]: Done   8 tasks      | elapsed:    6.9s
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:   12.6s
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:   17.3s
[Parallel(n_jobs=12)]: Done  37 tasks      | elapsed:   21.1s
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:   26.4s
[Parallel(n_jobs=12)]: Done  61 tasks      | elapsed:   32.2s
[Parallel(n_jobs=12)]: Done  74 tasks      | elapsed:   41.4s
[Parallel(n_jobs=12)]: Done  89 tasks      | elapsed:   47.5s
[Parallel(n_jobs=12)]: Done 104 tasks      | elapsed:   54.1s
[Parallel(n_jobs=12)]: Done 121 tasks      | elapsed:  1.0min
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed:  1.2min
[Parallel(n_jobs=12)]: Done 157 tasks      | elapsed:  1.3min
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:  1.4min
[Parallel(n_jobs=12)]: Done 197 tasks      | elapsed:  1.5min
[Parallel(n_jobs=12)]: Done 218 tasks      | elapsed:  1.6min
[Paralle

[TSTR] Saved full grid search results (4059 rows): /home/caineirb/Documents/DysCalc/DysCalc-ML-Development/outputs/grid_search/tstr_no_threshold_grid_search_results.csv

[TSTR] Top validation candidates:
   fbeta   recall  precision       f1  accuracy  conf_fact  min_samples_leaf  max_depth  tree_depth  tree_leaves
0.849057 0.857143   0.818182 0.837209   0.87037        0.1                10          6           6            7
0.849057 0.857143   0.818182 0.837209   0.87037        0.1                10          7           6            7
0.849057 0.857143   0.818182 0.837209   0.87037        0.1                11          6           6            7
0.849057 0.857143   0.818182 0.837209   0.87037        0.1                11          7           6            7
0.849057 0.857143   0.818182 0.837209   0.87037        0.1                12          6           6            7
0.849057 0.857143   0.818182 0.837209   0.87037        0.1                12          7           6            7
0.849

## PHASE 2 — TRTR vs TSTR Cross-Validation

TSTR folds only over real samples; synthetic rows are pinned to every training fold (Option A — no val leakage).



In [9]:
print('\n' + '=' * 60)
print('PHASE 2: TRTR vs TSTR CV')
print('=' * 60)

cv_metrics_trtr, fold_records_trtr = run_cv(
    X_train_trtr, y_train_trtr, 'TRTR', BEST_PARAMS_TRTR
)
cv_metrics_tstr, fold_records_tstr = run_cv(
    X_train_tstr, y_train_tstr, 'TSTR', BEST_PARAMS_TSTR,
    n_real=len(r_train)
)

print_cv_results('TRTR', cv_metrics_trtr)
print_cv_results('TSTR', cv_metrics_tstr)





PHASE 2: TRTR vs TSTR CV

Starting 5-Fold Cross-Validation [TRTR]...
  Fold 1: F1=0.6111, F2=0.5729, Recall=0.5500
  Fold 2: F1=0.5714, F2=0.5435, Recall=0.5263
  Fold 3: F1=0.6316, F2=0.6316, Recall=0.6316
  Fold 4: F1=0.4848, F2=0.4444, Recall=0.4211
  Fold 5: F1=0.7273, F2=0.7921, Recall=0.8421

Starting 5-Fold Cross-Validation [TSTR]...
  (folding over 250 real samples; 58 synthetic rows pinned to every train fold)
  Fold 1: F1=0.5778, F2=0.6190, Recall=0.6500
  Fold 2: F1=0.6667, F2=0.7353, Recall=0.7895
  Fold 3: F1=0.5455, F2=0.5000, Recall=0.4737
  Fold 4: F1=0.5641, F2=0.5729, Recall=0.5789
  Fold 5: F1=0.6667, F2=0.7071, Recall=0.7368

=== Cross-Validation Performance [TRTR] (5-Fold) ===
  Recall:    0.5942 +/- 0.1410
  Precision: 0.6311 +/- 0.0370
  F1-Score:  0.6052 +/- 0.0791
  F2-Score:  0.5969 +/- 0.1149
  Accuracy:  0.7120 +/- 0.0325

=== Cross-Validation Performance [TSTR] (5-Fold) ===
  Recall:    0.6458 +/- 0.1123
  Precision: 0.5797 +/- 0.0431
  F1-Score:  0.6041 +

## PHASE 3 — Side-by-side CV Summary



In [10]:
print('\n' + '=' * 60)
print('PHASE 3: CV Summary Comparison')
print('=' * 60)

metrics_labels = [
    ('Recall',    'mean_recall',    'std_recall'),
    ('Precision', 'mean_precision', 'std_precision'),
    ('F1-Score',  'mean_f1',        'std_f1'),
    ('F2-Score',  'mean_fbeta',     'std_fbeta'),
    ('Accuracy',  'mean_accuracy',  'std_accuracy'),
]

print(f'\n{"Metric":>12}  {"TRTR (mean+/-std)":>26}  {"TSTR (mean+/-std)":>26}  {"D (TSTR-TRTR)":>15}')
print('-' * 85)
for name, mean_key, std_key in metrics_labels:
    trtr_val = cv_metrics_trtr[mean_key]
    tstr_val = cv_metrics_tstr[mean_key]
    delta    = tstr_val - trtr_val
    sign     = '+' if delta >= 0 else ''
    print(f'{name:>12}  '
          f'{trtr_val:.4f} +/- {cv_metrics_trtr[std_key]:.4f}         '
          f'{tstr_val:.4f} +/- {cv_metrics_tstr[std_key]:.4f}         '
          f'{sign}{delta:.4f}')





PHASE 3: CV Summary Comparison

      Metric           TRTR (mean+/-std)           TSTR (mean+/-std)    D (TSTR-TRTR)
-------------------------------------------------------------------------------------
      Recall  0.5942 +/- 0.1410         0.6458 +/- 0.1123         +0.0516
   Precision  0.6311 +/- 0.0370         0.5797 +/- 0.0431         -0.0514
    F1-Score  0.6052 +/- 0.0791         0.6041 +/- 0.0521         -0.0011
    F2-Score  0.5969 +/- 0.1149         0.6269 +/- 0.0863         +0.0300
    Accuracy  0.7120 +/- 0.0325         0.6800 +/- 0.0358         -0.0320


## PHASE 4 — Fold Variance Inspection



In [11]:
print('\n' + '=' * 60)
print('PHASE 4: Fold Variance Inspection')
print('=' * 60)

print_variance_inspection('TRTR', fold_records_trtr)
print_variance_inspection('TSTR', fold_records_tstr)





PHASE 4: Fold Variance Inspection

--- Fold Variance Inspection [TRTR] ---
 Fold  N_train  C0_tr  C1_tr  N_val  C0_val  C1_val  Depth  Leaves   Recall       F2 Flag
-----------------------------------------------------------------------------------------------
    1      200    124     76     50      30      20      4       5   0.5500   0.5729
    2      200    123     77     50      31      19      6       8   0.5263   0.5435
    3      200    123     77     50      31      19      6       9   0.6316   0.6316
    4      200    123     77     50      31      19      4       5   0.4211   0.4444 LOW
    5      200    123     77     50      31      19      6       9   0.8421   0.7921 HIGH

  Recall range : 0.4211 - 0.8421
  Recall mean  : 0.5942 +/- 0.1576

  Correlation: class_1_train vs recall = +0.157
  Correlation: tree_depth     vs recall = +0.629

--- Fold Variance Inspection [TSTR] ---
 Fold  N_train  C0_tr  C1_tr  N_val  C0_val  C1_val  Depth  Leaves   Recall       F2 Flag
------

## PHASE 4b — TSTR Outlier Fold Investigation

Dynamically identifies the lowest-recall TSTR fold and
inspects the real at-risk cases in its val split to surface
subgroups the synthetic data may not be covering well.



In [12]:
# Detect outlier fold before printing header
tstr_fold_df     = pd.DataFrame(fold_records_tstr)
outlier_fold_no  = int(tstr_fold_df.loc[tstr_fold_df['recall'].idxmin(), 'fold'])
outlier_recall   = tstr_fold_df.loc[tstr_fold_df['recall'].idxmin(), 'recall']
mean_tstr_recall = tstr_fold_df['recall'].mean()

print('\n' + '=' * 60)
print(f'PHASE 4b: TSTR Fold {outlier_fold_no} Outlier Investigation')
print('=' * 60)

print(f'\n  Outlier fold: Fold {outlier_fold_no} '
      f'(recall={outlier_recall:.4f} vs mean={mean_tstr_recall:.4f})')

# Re-run the same split used in Phase 2 to recover val indices.
# CV in Phase 2 folded over real samples only — replicate that here.
X_real_inv = X_train_tstr.iloc[:len(r_train)].reset_index(drop=True)
y_real_inv = y_train_tstr.iloc[:len(r_train)].reset_index(drop=True)

skf_inv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_i, (_, val_idx) in enumerate(skf_inv.split(X_real_inv, y_real_inv), 1):
    if fold_i == outlier_fold_no:
        outlier_val_idx = val_idx
        break

# Recover real val rows for the outlier fold
real_val_fold_df = r_train.iloc[outlier_val_idx].copy()
score_cols       = ["NC", "DM", "NS", "ADD", "SUB", "CA"]

print(f'\n  Val split composition for Fold {outlier_fold_no}:')
print(f'    Real samples: {len(real_val_fold_df)}  '
      f'(class 0: {int((real_val_fold_df["Label"]==0).sum())}  '
      f'class 1: {int((real_val_fold_df["Label"]==1).sum())})')
print(f'    Synthetic samples: 0  (all synthetic pinned to train — Option A)')

real_pos_val = real_val_fold_df[real_val_fold_df['Label'] == 1]

if len(real_pos_val) > 0:
    print(f'\n  Real at-risk cases in Fold {outlier_fold_no} val (n={len(real_pos_val)}):')
    print(f'  {"Feature":>6} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
    print('  ' + '-' * 44)
    for col in score_cols:
        if col in real_pos_val.columns:
            vals = real_pos_val[col]
            print(f'  {col:>6} {vals.mean():>8.3f} {vals.std():>8.3f} '
                  f'{vals.min():>8.3f} {vals.max():>8.3f}')

    real_pos_train = r_train.iloc[
        [i for i in range(len(r_train)) if i not in outlier_val_idx]
    ]
    real_pos_train = real_pos_train[real_pos_train['Label'] == 1]

    if len(real_pos_train) > 0:
        print(f'\n  Real at-risk cases in Fold {outlier_fold_no} training (n={len(real_pos_train)}):')
        print(f'  {"Feature":>6} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
        print('  ' + '-' * 44)
        for col in score_cols:
            if col in real_pos_train.columns:
                vals = real_pos_train[col]
                print(f'  {col:>6} {vals.mean():>8.3f} {vals.std():>8.3f} '
                      f'{vals.min():>8.3f} {vals.max():>8.3f}')

    print(f'\n  If the val group has systematically different feature ranges,')
    print(f'  those subgroups are underrepresented in your synthetic data.')
else:
    print(f'\n  No real at-risk cases in Fold {outlier_fold_no} val split.')
    print(f'  Recall = 0 is expected — no positive examples to predict.')





PHASE 4b: TSTR Fold 3 Outlier Investigation

  Outlier fold: Fold 3 (recall=0.4737 vs mean=0.6458)

  Val split composition for Fold 3:
    Real samples: 50  (class 0: 31  class 1: 19)
    Synthetic samples: 0  (all synthetic pinned to train — Option A)

  Real at-risk cases in Fold 3 val (n=19):
  Feature     Mean      Std      Min      Max
  --------------------------------------------
      NC 1115.131  329.052  533.255 1639.529
      DM 2443.490  966.060 1212.225 4893.677
      NS   11.684    4.888    3.000   23.000
     ADD   29.895   13.420    6.000   54.000
     SUB   24.895   14.613    2.000   55.000
      CA   15.421    8.744    2.000   37.000

  Real at-risk cases in Fold 3 training (n=77):
  Feature     Mean      Std      Min      Max
  --------------------------------------------
      NC 1098.709  350.893  540.000 2050.356
      DM 2483.371  878.145 1065.750 4902.437
      NS   11.877    5.716    2.000   24.500
     ADD   31.390   13.070    5.000   61.000
     SUB   27.80

## PHASE 5 — Final Test Set Evaluation

Train on full TRTR / TSTR and evaluate the tree's native class predictions on the held-out test set.



In [13]:
print('\n' + '=' * 60)
print('PHASE 5: Test Set Evaluation')
print('=' * 60)

results = {}
params_map = {
    'TRTR': BEST_PARAMS_TRTR,
    'TSTR': BEST_PARAMS_TSTR
}
for label, X_tr, y_tr in [('TRTR', X_train_trtr, y_train_trtr),
                            ('TSTR', X_train_tstr, y_train_tstr)]:
    print(f'\n  [{label}] Locked params: {params_map[label]}')

    final_tree = C45DecisionTree(**params_map[label], feature_domain_mapping=FUNA_DB_DOMAIN_MAPPING)
    final_tree.fit(X_tr, y_tr, raw_features=FUNA_DB_RAW_FEATURES)

    test_preds      = get_preds(final_tree, X_test)
    m               = compute_metrics(y_test, test_preds)
    results[label]  = m

    # Keep TSTR final tree for Phase 6
    if label == 'TSTR':
        tstr_final_tree = final_tree

    print(f'\n  [{label}] Test Set Results:')
    print(f'    Recall:    {m["recall"]:.4f}')
    print(f'    Precision: {m["precision"]:.4f}')
    print(f'    F1-Score:  {m["f1"]:.4f}')
    print(f'    F2-Score:  {m["fbeta"]:.4f}')
    print(f'    Accuracy:  {m["accuracy"]:.4f}')

# Side-by-side test comparison
test_metrics_labels = [
    ('Recall',    'recall'),
    ('Precision', 'precision'),
    ('F1-Score',  'f1'),
    ('F2-Score',  'fbeta'),
    ('Accuracy',  'accuracy'),
]
print(f'\n{"Metric":>12} {"TRTR (test)":>14} {"TSTR (test)":>14} {"D (TSTR-TRTR)":>15}')
print('-' * 58)
for name, key in test_metrics_labels:
    trtr_val = results['TRTR'][key]
    tstr_val = results['TSTR'][key]
    delta    = tstr_val - trtr_val
    sign     = '+' if delta >= 0 else ''
    print(f'{name:>12}  {trtr_val:>12.4f}  {tstr_val:>12.4f}  {sign}{delta:.4f}')

# CV vs Test consistency check
print(f'\n--- CV vs Test Consistency Check ---')
for label, cv_m, test_m in [('TRTR', cv_metrics_trtr, results['TRTR']),
                              ('TSTR', cv_metrics_tstr, results['TSTR'])]:
    recall_drift = test_m['recall'] - cv_m['mean_recall']
    f2_drift     = test_m['fbeta']  - cv_m['mean_fbeta']
    sign_r = '+' if recall_drift >= 0 else ''
    sign_f = '+' if f2_drift     >= 0 else ''
    print(f'  [{label}]  Recall drift (test - CV mean): {sign_r}{recall_drift:.4f}  |  '
          f'F2 drift: {sign_f}{f2_drift:.4f}')
    if abs(recall_drift) > 0.10:
        print(f'    Recall drift > 0.10 -- possible overfitting or distribution mismatch.')





PHASE 5: Test Set Evaluation

  [TRTR] Locked params: {'conf_fact': 0.5, 'min_samples_leaf': 10, 'max_depth': 6}

  [TRTR] Test Set Results:
    Recall:    0.5238
    Precision: 0.6875
    F1-Score:  0.5946
    F2-Score:  0.5500
    Accuracy:  0.7222

  [TSTR] Locked params: {'conf_fact': 0.25, 'min_samples_leaf': 11, 'max_depth': 7}

  [TSTR] Test Set Results:
    Recall:    0.6667
    Precision: 0.6087
    F1-Score:  0.6364
    F2-Score:  0.6542
    Accuracy:  0.7037

      Metric    TRTR (test)    TSTR (test)   D (TSTR-TRTR)
----------------------------------------------------------
      Recall        0.5238        0.6667  +0.1429
   Precision        0.6875        0.6087  -0.0788
    F1-Score        0.5946        0.6364  +0.0418
    F2-Score        0.5500        0.6542  +0.1042
    Accuracy        0.7222        0.7037  -0.0185

--- CV vs Test Consistency Check ---
  [TRTR]  Recall drift (test - CV mean): -0.0704  |  F2 drift: -0.0469
  [TSTR]  Recall drift (test - CV mean): +0.020

## PHASE 6 — Global Feature Importance Analysis

Uses the TSTR final tree from Phase 5.
Split into (a) incomplete flags and (b) task features to
answer two questions:
1. Are incomplete flags genuinely used as split criteria?
2. Which task features drive the tree's decisions?



In [14]:
print('\n' + '=' * 60)
print('PHASE 6: Global Feature Importance (TSTR tree)')
print('=' * 60)

importance = tstr_final_tree.get_feature_importance()

flag_importance = {k: v for k, v in importance.items() if k in INCOMPLETE_FLAGS}
task_importance = {k: v for k, v in importance.items() if k not in INCOMPLETE_FLAGS}

flag_importance = dict(sorted(flag_importance.items(), key=lambda x: x[1], reverse=True))
task_importance = dict(sorted(task_importance.items(), key=lambda x: x[1], reverse=True))

print('\n  Incomplete flag importance (are they used as split nodes?):')
if flag_importance:
    for feat, score in flag_importance.items():
        bar = '#' * int(score * 40)
        print(f'    {feat:<20} {score:.4f}  {bar}')
    total_flag_share = sum(flag_importance.values())
    print(f'\n  -> Flags account for {total_flag_share:.2%} of total tree importance.')
    if total_flag_share > 0.05:
        print('  [OK] Incomplete flags carry meaningful split weight -- retain them as features.')
    else:
        print('  [i]  Flags have negligible split weight -- the tree largely ignores them.')
else:
    print('    (none -- the tree did not split on any incomplete flag)')
    print('  [i]  Incomplete flags were never chosen as split nodes.')

print('\n  Task feature importance (excluding incomplete flags):')
for feat, score in task_importance.items():
    bar = '#' * int(score * 40)
    print(f'    {feat:<20} {score:.4f}  {bar}')


FLAG_IMPORTANCE_CUTOFF = 0.01

used_flags   = [f for f, s in flag_importance.items() if s >= FLAG_IMPORTANCE_CUTOFF]
unused_flags = [f for f, s in flag_importance.items() if s <  FLAG_IMPORTANCE_CUTOFF]

print('\n' + '=' * 60)
print('PHASE 6b: Deployment CSV Export')
print('=' * 60)

if not flag_importance:
    # Tree never split on any flag — drop all of them
    unused_flags = list(INCOMPLETE_FLAGS)
    used_flags   = []
elif not unused_flags:
    print('\n  All incomplete flags carry meaningful split weight.')
    print('  No columns dropped — deployment CSVs not generated.')
else:
    print(f'\n  Unused incomplete flags (importance < {FLAG_IMPORTANCE_CUTOFF:.0%}):')
    for f in unused_flags:
        print(f'    - {f}  (importance={flag_importance.get(f, 0.0):.4f})')
    if used_flags:
        print(f'\n  Retained incomplete flags (importance >= {FLAG_IMPORTANCE_CUTOFF:.0%}):')
        for f in used_flags:
            print(f'    + {f}  (importance={flag_importance.get(f, 0.0):.4f})')

# ── Save deployment CSVs whenever there are flags to drop ──
if unused_flags:
    cols_to_drop = [f for f in unused_flags if f in r_train.columns]
    dataset_map  = {
        "train":   r_train,
        "s_train": s_train,
        "val":     val_df,
        "test":    test_df,
    }
    print(f'\n  Dropping {len(cols_to_drop)} unused flag column(s): {cols_to_drop}')
    print(f'  Saving deployment CSVs...')
    for name, df in dataset_map.items():
        drop_present = [c for c in cols_to_drop if c in df.columns]
        if not drop_present:
            print(f'    [{name}] No matching columns to drop — skipped.')
            continue
        deployment_df   = df.drop(columns=drop_present)
        deployment_path = DATASET_DIR / f"{name}_deployment.csv"
        deployment_df.to_csv(deployment_path, index=False)
        print(f'    [{name}] Saved → {deployment_path.name}  '
              f'(shape: {df.shape} → {deployment_df.shape})')
    print('\n  ✔ Deployment CSVs written to:', DATASET_DIR)





PHASE 6: Global Feature Importance (TSTR tree)

  Incomplete flag importance (are they used as split nodes?):
    (none -- the tree did not split on any incomplete flag)
  [i]  Incomplete flags were never chosen as split nodes.

  Task feature importance (excluding incomplete flags):
    NC                   0.4928  ###################
    NS                   0.2584  ##########
    ADD                  0.1876  #######
    SUB                  0.0612  ##

PHASE 6b: Deployment CSV Export

  Dropping 6 unused flag column(s): ['NS_incomplete', 'SUB_incomplete', 'CA_incomplete', 'NC_incomplete', 'DM_incomplete', 'ADD_incomplete']
  Saving deployment CSVs...
    [train] Saved → train_deployment.csv  (shape: (250, 19) → (250, 13))
    [s_train] Saved → s_train_deployment.csv  (shape: (58, 19) → (58, 13))
    [val] Saved → val_deployment.csv  (shape: (54, 19) → (54, 13))
    [test] Saved → test_deployment.csv  (shape: (54, 19) → (54, 13))

  ✔ Deployment CSVs written to: /home/caineirb/Docum